In [ ]:
"""ЗАДАНИЕ 3.1. Загрузка и сохранение файла из списка путей Common Crawl"""
import requests
import gzip
import os

def download_and_extract_paths(url: str, local_gz_path: str) -> list:
    """Скачивает gzip-файл со списком путей и распаковывает его построчно."""
    print(f"Скачиваем список путей из {url}...")
    response = requests.get(url)
    response.raise_for_status()

    with open(local_gz_path, "wb") as f:
        f.write(response.content)
    print(f"Список путей сохранен как {local_gz_path}")

    with gzip.open(local_gz_path, "rt") as f:
        paths = [line.strip() for line in f if line.strip()]
    print(f"Найдено путей: {len(paths)}")
    return paths

def download_file(base_url: str, path: str, save_dir: str) -> str:
    """Скачивает файл по URL и сохраняет в указанную папку."""
    os.makedirs(save_dir, exist_ok=True)
    file_name = os.path.basename(path)
    save_path = os.path.join(save_dir, file_name)

    full_url = base_url + path
    print(f"Скачиваем {file_name}...")

    with requests.get(full_url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    size_mb = os.path.getsize(save_path) / 1024 / 1024
    print(f"Готово! {file_name} сохранен в {save_path} ({size_mb:.2f} MB)")
    return save_path

# ---- Используем функции ----

url_paths = "https://data.commoncrawl.org/crawl-data/CC-MAIN-2020-40/warc.paths.gz"
local_gz = "warc.paths.gz"
base_url = "https://data.commoncrawl.org/"
save_folder = "/content/drive/MyDrive/CommonCrawl/"

# 1. Получаем список путей
paths = download_and_extract_paths(url_paths, local_gz)

# 2. Выбираем первый файл для загрузки
chosen_path = paths[0]

# 3. Скачиваем файл в указанную папку
download_file(base_url, chosen_path, save_folder)

Скачиваем список путей из https://data.commoncrawl.org/crawl-data/CC-MAIN-2020-40/warc.paths.gz...
Список путей сохранен как warc.paths.gz
Найдено путей: 79600
Скачиваем CC-MAIN-20200918061627-20200918091627-00000.warc.gz...
Готово! CC-MAIN-20200918061627-20200918091627-00000.warc.gz сохранен в /content/drive/MyDrive/CommonCrawl/CC-MAIN-20200918061627-20200918091627-00000.warc.gz (1479.22 MB)


'/content/drive/MyDrive/CommonCrawl/CC-MAIN-20200918061627-20200918091627-00000.warc.gz'

In [ ]:
"""ЗАДАНИЕ 3.1 Загрузка файла (чтение для конвертации в текст)"""
# Устанавливаем библиотеку для чтения WARC архивов (если ещё не установлена)
!pip install warcio -q

from warcio.archiveiterator import ArchiveIterator

def read_warc_file(filepath: str, max_pages: int = 5):
    """
    Читает WARC-файл и выводит информацию о первых max_pages HTML-страницах.

    Args:
        filepath (str): Путь к WARC-файлу.
        max_pages (int): Максимальное количество страниц для чтения.
    """
    count = 0
    print(f"Чтение файла: {filepath}")

    try:
        with open(filepath, 'rb') as stream:
            for record in ArchiveIterator(stream):
                # Ищем ответы сервера (HTML страницы)
                if record.rec_type == 'response':
                    url = record.rec_headers.get_header('WARC-Target-URI')
                    print(f"Найдена страница: {url}")

                    content = record.content_stream().read()
                    print(f"Размер HTML: {len(content)} байт")
                    print("-" * 30)

                    count += 1
                    if count >= max_pages:
                        print(f"Готово! Прочитано {max_pages} страницы.")
                        break
    except FileNotFoundError:
        print("Ошибка: Файл не найден. Проверьте путь к файлу.")
    except Exception as e:
        print(f"Произошла ошибка при чтении файла: {e}")

# Пример вызова функции с нужным путем
filename = "/content/drive/MyDrive/CommonCrawl/CC-MAIN-20200918061627-20200918091627-00000.warc.gz"
read_warc_file(filename)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.8 MB/s eta 0:00:00
Чтение файла: /content/drive/MyDrive/CommonCrawl/CC-MAIN-20200918061627-20200918091627-00000.warc.gz
Найдена страница: http://00.fryehj.cn/9t0y8i3f02.xml
Размер HTML: 38579 байт
------------------------------
Найдена страница: http://002581.cn/report/
Размер HTML: 10176 байт
------------------------------
Найдена страница: http://0035.blog85.fc2.com/blog-entry-1205.html
Размер HTML: 6855 байт
------------------------------
Найдена страница: http://01.baojv.cn/bhjnc1j7ae.xml
Размер HTML: 12868 байт
------------------------------
Найдена страница: http://01.hkwordpress.com/delicious-hot-grilled-chicken-recipes/
Размер HTML: 63016 байт
------------------------------
Готово! Прочитано 5 страницы.


In [ ]:
"""ЗАДАНИЕ 3.1 Загрузка файла, очистка от заголовков"""

from bs4 import BeautifulSoup

def clean_html(html: str) -> str:
    """Очищает HTML от тегов и возвращает очищенный текст."""
    soup = BeautifulSoup(html, 'html.parser')
    # Удаляем ненужные теги
    for tag in soup(["script", "style", "header", "footer", "nav"]):
        tag.decompose()
    text = soup.get_text(separator=' ')
    # Сжимаем пробелы
    return " ".join(text.split())

def warc_to_text(input_path: str, output_path: str, max_pages: int = 2000) -> int:
    """Конвертирует WARC файл в текстовый, очищенный от HTML тегов.

    Args:
        input_path: путь к WARC файлу
        output_path: путь сохранения результата
        max_pages: максимальное число обрабатываемых страниц

    Returns:
        Количество преобразованных страниц
    """
    count = 0
    print("Начинаем конвертацию WARC в текст...")

    with open(output_path, 'w', encoding='utf-8') as out_f:
        with open(input_path, 'rb') as stream:
            for record in ArchiveIterator(stream):
                if record.rec_type == 'response':
                    try:
                        html_content = record.content_stream().read().decode('utf-8', errors='ignore')
                        cleaned_text = clean_html(html_content)
                        url = record.rec_headers.get_header('WARC-Target-URI')

                        out_f.write(f"URL: {url}\n")
                        out_f.write(cleaned_text)
                        out_f.write("\n\n--- NEXT PAGE ---\n\n")

                        count += 1
                        if count >= max_pages:
                            print(f"Достигнут лимит в {max_pages} страниц.")
                            break
                    except Exception:
                        # Можем добавить лог ошибок, если нужно
                        continue

    final_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Обработано страниц: {count}")
    print(f"Результат сохранен в: {output_path} ({final_size_mb:.2f} MB)")
    return count

# Использование функции
input_file = "/content/drive/MyDrive/CommonCrawl/CC-MAIN-20200918061627-20200918091627-00000.warc.gz"
output_txt = "/content/raw_data.txt"

warc_to_text(input_file, output_txt, max_pages=2000)

Начинаем конвертацию WARC в текст...


/tmp/ipykernel_1713/1779412481.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, 'html.parser')


Достигнут лимит в 2000 страниц.
Обработано страниц: 2000
Результат сохранен в: /content/raw_data.txt (18.03 MB)


2000

In [ ]:
"""ЗАДАНИЕ 3.2 Очистка данных"""
import re
import shutil

def clean_text(raw_text: str, min_length: int = 20, chunk_word_size: int = 500, min_chunk_length: int = 100):
    """
    Очищает текст: убирает пустые строки, оставляет только разрешённые символы,
    фильтрует по длине, разбивает на чанки.

    Args:
        raw_text (str): исходный текст.
        min_length (int): минимальная длина строки для сохранения.
        chunk_word_size (int): количество слов в одном чанке.
        min_chunk_length (int): минимальная длина чанка в символах.

    Returns:
        list[str]: список очищенных текстовых чанков.
    """
    allowed_pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9 .,!?;:()\-«»"\'\n]'

    # Разбиваем на строки, удаляем пустые и нормализуем пробелы
    lines = [line.strip() for line in raw_text.split('\n') if line.strip()]

    filtered = []
    for line in lines:
        line = " ".join(line.split())  # нормализация пробелов
        line = re.sub(allowed_pattern, '', line)  # удаляем запрещённые символы
        if len(line) > min_length:
            filtered.append(line)

    cleaned_text = '\n'.join(filtered)

    # Разбиваем на чанки по заданному числу слов
    words = cleaned_text.split()
    chunks = []
    for i in range(0, len(words), chunk_word_size):
        chunk = ' '.join(words[i:i + chunk_word_size])
        if len(chunk) > min_chunk_length:
            chunks.append(chunk)

    return chunks

def main():
    input_path = "/content/raw_data.txt"
    output_path = "/content/cleaned_dataset.txt"
    drive_path = "/content/drive/MyDrive/CommonCrawl/cleaned_dataset.txt"

    print("Начинаем очистку данных (3.2)...")

    with open(input_path, 'r', encoding='utf-8') as f:
        raw_text = f.read()

    chunks = clean_text(raw_text)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(chunks))

    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

    print(f"Очистка завершена")
    print(f"Обработано объектов (чанков): {len(chunks)}")
    print(f"Размер очищенного файла: {file_size_mb:.2f} MB")

    shutil.copy(output_path, drive_path)
    print(f"Файл успешно сохранён на Google Drive: {drive_path}")

if __name__ == "__main__":
    main()

Начинаем очистку данных (3.2)...
Очистка завершена
Обработано объектов (чанков): 3580
Размер очищенного файла: 12.27 MB
Файл успешно сохранён на Google Drive: /content/drive/MyDrive/CommonCrawl/cleaned_dataset.txt


In [ ]:
"""ЗАДАНИЕ 3.3 Удаление дубликатов и подсчет энтропии с GPT-2"""
!pip install transformers torch -q

import torch
import numpy as np
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from tqdm import tqdm

def load_chunks(file_path):
    """Загрузить чанки из файла, разделённые двойным переносом строки."""
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]

def init_model(model_name="gpt2", device=None):
    """Инициализация модели и токенизатора GPT-2 с настройкой pad_token."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    model.eval()
    return model, tokenizer, device

def calculate_entropy(texts, model, tokenizer, device, batch_size=2):
    """Вычисляет сумму энтропии для каждого текста в списке батчами."""
    entropies = []
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')

    for i in tqdm(range(0, len(texts), batch_size), desc="Подсчет энтропии"):
        batch = texts[i:i+batch_size]
        encodings = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = encodings['input_ids'][..., 1:].contiguous()

            losses = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            losses = losses.view(shift_labels.size(0), -1)
            mask = encodings['attention_mask'][..., 1:].contiguous()

            for j in range(len(batch)):
                valid_losses = losses[j][mask[j] == 1]
                entropies.append(valid_losses.sum().item())
    return entropies

def filter_chunks(chunks, entropies):
    """Удаляет дубликаты и фильтрует по порогам энтропии (5-й и 95-й процентили)."""
    # Удаляем дубликаты, сохраняя порядок
    seen = set()
    unique_chunks = []
    for c in chunks:
        if c not in seen:
            unique_chunks.append(c)
            seen.add(c)
    print(f"Удалено дубликатов: {len(chunks) - len(unique_chunks)}")

    entropies_np = np.array(entropies)
    low_p = np.percentile(entropies_np, 5)
    high_p = np.percentile(entropies_np, 95)
    print(f"Порог низкой энтропии (5%): {low_p:.2f}")
    print(f"Порог высокой энтропии (95%): {high_p:.2f}")

    filtered = []
    seen_filtered = set()
    for c, ent in zip(chunks, entropies):
        if c in seen_filtered:
            continue
        if low_p <= ent <= high_p:
            filtered.append(c)
            seen_filtered.add(c)
    return filtered

def compute_information_density(chunks, entropies, tokenizer):
    """Считает информационную плотность rho_info (энтропия на токен)."""
    total_entropy = sum(entropies)
    total_tokens = sum(len(tokenizer.encode(c)) for c in chunks)
    return total_entropy / total_tokens if total_tokens > 0 else 0

def main():
    input_path = "/content/drive/MyDrive/CommonCrawl/cleaned_dataset.txt"
    output_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset.txt"

    print("Загрузка данных и подготовка...")
    chunks = load_chunks(input_path)
    print(f"Исходное число объектов: {len(chunks)}")

    model, tokenizer, device = init_model()
    print(f"Используем устройство: {device}")

    print("Подсчет энтропии по батчам...")
    entropies = calculate_entropy(chunks, model, tokenizer, device, batch_size=2)

    filtered_chunks = filter_chunks(chunks, entropies)

    rho_info = compute_information_density(chunks, entropies, tokenizer)
    print(f"\nИнформационная плотность (бит/токен): {rho_info:.4f}")
    print(f"Оставлено объектов после фильтрации: {len(filtered_chunks)} из {len(chunks)}")

    print(f"Сохраняем отфильтрованный датасет в {output_path} ...")
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(filtered_chunks))

    print("Готово!")

if __name__ == "__main__":
    main()

Загрузка данных и подготовка...
Исходное число объектов: 3580


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Используем устройство: cuda
Подсчет энтропии по батчам...


Подсчет энтропии: 100%|██████████| 1790/1790 [05:50<00:00,  5.11it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1069 > 1024). Running this sequence through the model will result in indexing errors


Удалено дубликатов: 0
Порог низкой энтропии (5%): 741.65
Порог высокой энтропии (95%): 6179.92

Информационная плотность (бит/токен): 2.0680
Оставлено объектов после фильтрации: 3222 из 3580
Сохраняем отфильтрованный датасет в /content/drive/MyDrive/CommonCrawl/filtered_dataset.txt ...
Готово!


In [ ]:
"""ПОДСЧЕТ ЧАСТИЧНЫХ ДУБЛИКАТОВ (N-грамм)"""
from collections import Counter

# 1. Загружаем отфильтрованный датасет
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
print(f"Количество объектов для анализа: {len(chunks)}")

# 2. Собираем n-граммы (последовательности из N слов)
N = 20  # Длина n-граммы
ngrams = []

for chunk in chunks:
    words = chunk.split()
    if len(words) < N:
        continue  # пропускаем очень короткие объекты
    for i in range(len(words) - N + 1):
        ngram = ' '.join(words[i:i+N])
        ngrams.append(ngram)

print(f"Общее количество n-грамм длиной {N}: {len(ngrams)}")

# 3. Считаем частоты встречаемости n-грамм
ngram_counts = Counter(ngrams)

# 4. Анализ повторов
total_ngrams = len(ngrams)
unique_ngrams = len(ngram_counts)
duplicated_ngrams = sum(count for count in ngram_counts.values() if count > 1)

# 5. Дублирующая плотность (rho_dub) — как часто n-граммы повторяются
rho_dub = total_ngrams / unique_ngrams if unique_ngrams else 0

print(f"\n--- Итоги анализа дубликатов для n={N} ---")
print(f"Всего n-грамм: {total_ngrams}")
print(f"Уникальных n-грамм: {unique_ngrams}")
print(f"Повторяющихся n-грамм: {duplicated_ngrams}")
print(f"Дублирующая плотность (rho_dub): {rho_dub:.4f}")

# 6. Оценка качества по плотности дубликатов
if rho_dub < 1.05:
    print("\nДатасет очень чистый — практически без дубликатов.")
elif rho_dub < 1.2:
    print("\nУмеренное количество дубликатов — типично для Common Crawl.")
else:
    print("\nВ датасете много дубликатов — рекомендована дополнительная очистка.")

Количество объектов для анализа: 3222
Общее количество n-грамм длиной 20: 1549716

--- Итоги анализа дубликатов для n=20 ---
Всего n-грамм: 1549716
Уникальных n-грамм: 1442868
Повторяющихся n-грамм: 165040
Дублирующая плотность (rho_dub): 1.0741

Умеренное количество дубликатов — типично для Common Crawl.


In [ ]:
"""ТОКЕНИЗАЦИЯ ПО СИМВОЛАМ"""
import random

# 1. Загружаем очищенный датасет
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
print(f"Загружено объектов: {len(chunks)}")

# 2. Собираем множество всех уникальных символов из всего датасета
all_chars = set(char for chunk in chunks for char in chunk)

# 3. Создаём словари отображения символ <-> индекс
vocab = sorted(all_chars)  # сортируем для стабильности индексации
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

print(f"\n--- Результаты токенизации на уровне символов ---")
print(f"Размер словаря (уникальных символов): {len(vocab)}")
print(f"Первые 20 символов из словаря: {vocab[:20]}")

# 4. Выбираем случайный объект для демонстрации
random_chunk = random.choice(chunks)
print(f"\nСлучайный объект (первые 200 символов):\n'{random_chunk[:200]}...'")

# 5. Токенизируем выбранный объект
tokenized = [char_to_idx[char] for char in random_chunk]
print(f"\nДлина последовательности (число токенов): {len(tokenized)}")
print(f"Первые 50 токенов: {tokenized[:50]}")

# 6. Декодируем обратно и проверяем корректность
decoded = ''.join(idx_to_char[idx] for idx in tokenized)
print("\nПроверка обратного декодирования:")
print(f"Совпадает с оригиналом: {decoded == random_chunk}")

Загружено объектов: 3222

--- Результаты токенизации на уровне символов ---
Размер словаря (уникальных символов): 142
Первые 20 символов из словаря: [' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':']

Случайный объект (первые 200 символов):
'terms United States of America Andorra Afghanistan Antigua Barbuda Anguilla Albania Armenia Netherlands Antilles Angola Antarctica Argentina American Samoa Austria Australia Aruba Azerbaijan Bosnia an...'

Длина последовательности (число токенов): 3477
Первые 50 токенов: [67, 52, 65, 60, 66, 0, 42, 61, 56, 67, 52, 51, 0, 40, 67, 48, 67, 52, 66, 0, 62, 53, 0, 22, 60, 52, 65, 56, 50, 48, 0, 22, 61, 51, 62, 65, 65, 48, 0, 22, 53, 54, 55, 48, 61, 56, 66, 67, 48, 61]

Проверка обратного декодирования:
Совпадает с оригиналом: True


In [ ]:
"""ДОПОЛНИТЕЛЬНАЯ ФИЛЬТРАЦИЯ ДЛЯ ТОКЕНИЗАЦИИ ПО СЛОВАМ"""

# Загружаем датасет
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset.txt"
output_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
print(f"Объектов до фильтрации: {len(chunks)}")

def is_good_text(chunk):
    """Эвристика для определения, является ли текст связным и качественным."""
    # Проверка длины текста и количества слов
    if len(chunk) < 100:
        return False
    words = chunk.split()
    if len(words) < 20:
        return False

    # Доля цифр в тексте не должна превышать 30%
    digits_count = sum(c.isdigit() for c in chunk)
    if digits_count / len(chunk) > 0.3:
        return False

    # Доля знаков препинания не должна превышать 30%
    punctuation_marks = '.,!?;:()[]{}"\''
    punctuation_count = sum(c in punctuation_marks for c in chunk)
    if punctuation_count / len(chunk) > 0.3:
        return False

    # Отсеиваем тексты с 4 и более подряд идущими одинаковыми знаками препинания
    if re.search(r'([!?.,])\1{3,}', chunk):
        return False

    # Проверяем долю уникальных слов — слишком высокая (90%+) может указывать на списки или технический текст
    unique_words = len(set(w.lower() for w in words))
    if unique_words / len(words) > 0.9:
        return False

    # Проверяем строки, начинающиеся с дефиса, звездочки или цифры — более 50% таких строк говорит о списках
    lines = chunk.split('\n')
    if len(lines) > 0:
        list_like_lines = sum(bool(re.match(r'^\s*[-*\d]', line)) for line in lines)
        if list_like_lines / len(lines) > 0.5:
            return False

    return True

# Фильтрация чанков
clean_chunks = [c for c in chunks if is_good_text(c)]

print(f"Объектов после фильтрации: {len(clean_chunks)}")
print(f"Удалено объектов как 'мусор': {len(chunks) - len(clean_chunks)}")

# Сохраняем результат
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(clean_chunks))
print(f"Отфильтрованный датасет сохранён: {output_path}")

Объектов до фильтрации: 3222
Объектов после фильтрации: 2230
Удалено объектов как 'мусор': 992
Отфильтрованный датасет сохранён: /content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt


In [ ]:
"""ТОКЕНИЗАЦИЯ ПО СЛОВАМ"""

# 1. Загружаем очищенный датасет
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
print(f"Загружено объектов: {len(chunks)}")

# 2. Собираем все уникальные слова из всего датасета
all_words = []
for chunk in chunks:
    words = chunk.lower().split()
    all_words.extend(words)

# 3. Считаем частоты встречаемости слов
word_counts = Counter(all_words)

# 4. Формируем словарь с фильтром по частоте
min_freq = 2
vocab = ['<PAD>', '<UNK>'] + sorted([w for w, c in word_counts.items() if c >= min_freq])
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

print(f"\n--- Результаты токенизации на уровне слов ---")
print(f"Общее количество слов в датасете: {len(all_words)}")
print(f"Уникальных слов (до фильтрации): {len(word_counts)}")
print(f"Размер словаря (с min_freq={min_freq}): {len(vocab)}")
print(f"Пример слов из словаря (с 200-го по 210-й): {vocab[200:210]}")

# 5. Демонстрация токенизации случайного объекта
random_chunk = random.choice(chunks)
print(f"\nСлучайный объект (первые 200 символов):\n'{random_chunk[:200]}...'")

words = random_chunk.lower().split()
tokenized = [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]

print(f"\nДлина последовательности (число токенов): {len(tokenized)}")
print(f"Первые 20 токенов: {tokenized[:20]}")

# 6. Проверка обратного декодирования
decoded_words = [idx_to_word[idx] for idx in tokenized]
decoded_text = ' '.join(decoded_words)

print(f"\nПроверка декодирования (первые 200 символов):\n'{decoded_text[:200]}...'")

# 7. Статистика неизвестных слов (OOV)
oov_count = sum(1 for word in words if word not in word_to_idx)
print(f"\nНеизвестных (OOV) слов: {oov_count} из {len(words)} ({(oov_count / len(words)) * 100:.2f}%)")

Загружено объектов: 2230

--- Результаты токенизации на уровне слов ---
Общее количество слов в датасете: 1114934
Уникальных слов (до фильтрации): 221466
Размер словаря (с min_freq=2): 75418
Пример слов из словаря (с 200-го по 210-й): ['"бектемн"', '"бумеранг"', '"гк', '"готовиться', '"граната', '"десертные', '"детское', '"длет"', '"елочка"', '"заря"']

Случайный объект (первые 200 символов):
'Heaven A HeavenGames Gaming Community Welcome Downloads Home Best Files Review Guidelines Main site Forums Code of Conduct Search Advanced Search Single Player Scenarios Home New Releases New Reviews ...'

Длина последовательности (число токенов): 500
Первые 20 токенов: [28259, 8370, 28261, 26287, 17407, 57703, 21483, 28826, 13276, 24962, 46931, 27596, 35460, 49935, 25592, 17071, 40047, 17716, 48609, 9084]

Проверка декодирования (первые 200 символов):
'heaven a heavengames gaming community welcome downloads home best files review guidelines main site forums code of conduct search advanced search

In [ ]:
"""BPE-ТОКЕНИЗАЦИЯ С ИСПОЛЬЗОВАНИЕМ БИБЛИОТЕКИ"""
# Установка библиотеки tokenizers (если ещё не установлена)
!pip install tokenizers -q

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Загружаем датасет (v2 — после дополнительной очистки)
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
print(f"Загружено объектов: {len(chunks)}")

# 2. Инициализируем BPE-токенизатор с указанным токеном для OOV слов
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# Прето-токенизация: разбиваем текст на токены по пробелам (whitespace)
tokenizer.pre_tokenizer = Whitespace()

# 3. Настраиваем и запускаем обучение токенизатора
trainer = BpeTrainer(
    vocab_size=20000,
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"],
    min_frequency=2
)

print("Обучение BPE-токенизатора...")
tokenizer.train_from_iterator(chunks, trainer=trainer)

print(f"\n--- Результаты BPE-токенизации ---")
print(f"Размер обученного словаря: {tokenizer.get_vocab_size()}")

# 4. Выбираем случайный объект для демонстрации
random_chunk = random.choice(chunks)
print(f"\nСлучайный объект (первые 200 символов):")
print(f"'{random_chunk[:200]}...'")

# 5. Токенизируем выбранный объект
encoding = tokenizer.encode(random_chunk)
print(f"\nДлина последовательности (число токенов): {len(encoding.ids)}")
print(f"Первые 20 токенов (ID): {encoding.ids[:20]}")
print(f"Первые 20 токенов (текст): {encoding.tokens[:20]}")

# 6. Обратное декодирование для проверки качества
decoded = tokenizer.decode(encoding.ids)
print(f"\nПроверка декодирования (первые 200 символов):")
print(f"'{decoded[:200]}...'")
print(f"Совпадает с оригиналом: {decoded == random_chunk}")

# 7. Демонстрация работы с незнакомым словом
test_word = "Hello,world!"
encoded_test = tokenizer.encode(test_word)
print(f"\n--- Тест на незнакомом слове: '{test_word}' ---")
print(f"Разбивка на токены: {encoded_test.tokens}")

Загружено объектов: 2230
Обучение BPE-токенизатора...

--- Результаты BPE-токенизации ---
Размер обученного словаря: 20000

Случайный объект (первые 200 символов):
'D Airs d'Italie Cline Dion Coups de Coeur Otis Redding Shadows Derniers visiteurs Statistiques Visiteurs depuis le 08012014 : 101077 Connects : 1 Record de connects : 574 Visiteurs du Monde Mto Mto Al...'

Длина последовательности (число токенов): 998
Первые 20 токенов (ID): [28, 2773, 69, 54, 6, 4187, 228, 27, 531, 28, 176, 5386, 572, 163, 875, 3569, 7148, 156, 2724, 1622]
Первые 20 токенов (текст): ['D', 'Air', 's', 'd', "'", 'Ital', 'ie', 'C', 'line', 'D', 'ion', 'Cou', 'ps', 'de', 'Co', 'eur', 'Ot', 'is', 'Red', 'ding']

Проверка декодирования (первые 200 символов):
'D Air s d ' Ital ie C line D ion Cou ps de Co eur Ot is Red ding Sh adow s Der ni ers vis ite urs St atis tiques Vis ite urs depuis le 08 01 2014 : 10 1077 Conn ects : 1 Record de conn ects : 57 4 Vis...'
Совпадает с оригиналом: False

--- Тест на незнакомо

In [ ]:
"""РУЧНАЯ BPE-ТОКЕНИЗАЦИЯ"""
from collections import Counter, defaultdict

# 1. Загружаем датасет
input_path = "/content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt"

with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = [c.strip() for c in text.split('\n\n') if c.strip()]
print(f"Загружено объектов: {len(chunks)}")

# 2. Берем подвыборку для обучения (BPE с нуля медленный на больших данных)
# Возьмем первые 200 объектов для демонстрации
train_chunks = chunks[:200]
print(f"Обучение BPE на {len(train_chunks)} объектах...")

# 3. Подготовка: разбиваем текст на слова и представляем каждое слово как список символов + маркер конца слова
# Например, "кот" -> ['к', 'о', 'т', '</w>']
def prepare_corpus(texts):
    word_freqs = Counter()
    for text in texts:
        for word in text.lower().split():
            # Добавляем маркер конца слова, чтобы BPE знал, где заканчивается слово
            word_freqs[tuple(word) + ('</w>',)] += 1
    return word_freqs

word_freqs = prepare_corpus(train_chunks)
print(f"Уникальных слов в подвыборке: {len(word_freqs)}")

# 4. Функция для получения всех пар символов
def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

# 5. Функция для объединения пары во всех словах
def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        # Проходим по слову и заменяем пару на объединенный токен
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(replacement)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

# 6. Основной цикл BPE
num_merges = 500  # Сколько раз объединять пары. Чем больше — тем крупнее токены.
vocab = dict(word_freqs)
merges = []  # Список правил объединения (для декодирования)

print(f"\n--- Обучение BPE ({num_merges} операций объединения) ---")
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    if (i + 1) % 100 == 0:
        print(f"  Итерация {i+1}/{num_merges}: объединяем {best_pair} (частота {pairs[best_pair]})")

# 7. Формируем итоговый словарь токенов
vocab_tokens = set()
for word in vocab.keys():
    for token in word:
        vocab_tokens.add(token)

print(f"\n--- Результаты чистой BPE-реализации ---")
print(f"Размер словаря: {len(vocab_tokens)}")
print(f"Примеры токенов: {list(vocab_tokens)[:20]}")

# 8. Функция токенизации с использованием выученных merges
def bpe_tokenize(word, merges):
    """Применяет выученные правила к новому слову"""
    tokens = list(word.lower()) + ['</w>']
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(''.join(pair))
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

# 9. Тестируем на случайном объекте
random_chunk = random.choice(chunks)
print(f"\nСлучайный объект (первые 200 символов):")
print(f"'{random_chunk[:200]}...'")

# Токенизируем первые 30 слов из объекта
words_to_test = random_chunk.split()[:30]
all_tokens = []
for word in words_to_test:
    tokens = bpe_tokenize(word, merges)
    all_tokens.extend(tokens)

print(f"\nТокенизировано слов: {len(words_to_test)}")
print(f"Получено токенов: {len(all_tokens)}")
print(f"Первые 30 токенов: {all_tokens[:30]}")

# 10. Тест на незнакомом слове
test_word = "токенизацияпрекрасна"
print(f"\n--- Тест на незнакомом слове: '{test_word}' ---")
print(f"Токены: {bpe_tokenize(test_word, merges)}")

Загружено объектов: 1834
Обучение BPE на 200 объектах...
Уникальных слов в подвыборке: 34177

--- Обучение BPE (500 операций объединения) ---
  Итерация 100/500: объединяем ('v', '</w>') (частота 707)
  Итерация 200/500: объединяем ('in', 'e</w>') (частота 323)
  Итерация 300/500: объединяем ('j', 'u') (частота 208)
  Итерация 400/500: объединяем ('th', 'er</w>') (частота 152)
  Итерация 500/500: объединяем ('p', 'o</w>') (частота 118)

--- Результаты чистой BPE-реализации ---
Размер словаря: 582
Примеры токенов: ['ор', 'we', "'</w>", 'all', 'wor', 'fu', 'ti', 'en,</w>', 'mar', '4)</w>', 'ed</w>', 'me</w>', 'ac', '(', 'de', '?</w>', 'http', '...</w>', '7)</w>', 'ma']

Случайный объект (первые 200 символов):
', ISIS». : http:notismarias.gr BlogThis! Twitter Facebook Pinterest 2014: 25 . . . . 40 . . . . , 162 . .. . 25 . . 19 19:30 4, 2014: , http:ardin-rixi.grarchives17498 15 18:00 3 - 2 , 12 19:00 13 18:...'

Токенизировано слов: 30
Получено токенов: 67
Первые 30 токенов: [',</w>', 'i

In [ ]:
!pip install datasets -q

"""Скачивание датасета wikitext"""

from datasets import load_dataset

# Загружаем датасет wikitext.
# "Salesforce/wikitext" — это официальный репозиторий.
print("Загрузка датасета wikitext...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

print(f"\n--- Структура датасета ---")
print(dataset)

# Смотрим на примеры
print(f"\n--- Пример из train ---")
print(dataset['train'][0])
print(f"\n--- Пример из validation ---")
print(dataset['validation'][0])
print(f"\n--- Пример из test ---")
print(dataset['test'][0])

# Считаем размеры
print(f"\n--- Размеры выборок ---")
print(f"Train: {len(dataset['train'])} строк")
print(f"Validation: {len(dataset['validation'])} строк")
print(f"Test: {len(dataset['test'])} строк")

Загрузка датасета wikitext...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


--- Структура датасета ---
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

--- Пример из train ---
{'text': ''}

--- Пример из validation ---
{'text': ''}

--- Пример из test ---
{'text': ''}

--- Размеры выборок ---
Train: 36718 строк
Validation: 3760 строк
Test: 4358 строк


In [ ]:
from datasets import load_dataset

# 1. Загружаем датасет wikitext, версия wikitext-2-raw-v1
print("Загрузка датасета wikitext...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# 2. Берём подвыборку из train — первые N_LINES строк
N_LINES = 50000
train_lines = dataset['train']['text'][:N_LINES]
print(f"Выбрано строк из train: {len(train_lines)}")

# 3. Формируем объекты (абзацы) из строк
# Логика: собираем непустые строки подряд, пустая строка — разделитель абзацев
chunks = []
current_chunk = []

for line in train_lines:
    line = line.strip()
    if not line:
        # Конец текущего абзаца
        if current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
    else:
        # Пропускаем заголовки (начинаются с "=")
        if not line.startswith("="):
            current_chunk.append(line)

# Добавляем последний абзац, если остался
if current_chunk:
    chunks.append(" ".join(current_chunk))

print(f"Сформировано объектов: {len(chunks)}")

# 4. Выводим несколько примеров для проверки
print("\n--- Примеры объектов ---")
for i in range(min(3, len(chunks))):
    print(f"[{i}] {chunks[i][:200]}...\n")

Загрузка датасета wikitext...
Выбрано строк из train: 36718
Сформировано объектов: 5487

--- Примеры объектов ---
[0] Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ pl...

[1] As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces . ...

[2] The game takes place during the Second Europan War . Gallian Army Squad 422 , also known as " The Nameless " , are a penal military unit composed of criminals , foreign deserters , and military offend...



In [ ]:
"""
Пункт 3.5: Применяем пайплайн 3.2 и 3.3 к wikitext
"""

from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from tqdm import tqdm

# ============================================================
# 3.2: ОЧИСТКА ТЕКСТА WIKITEXT
# ============================================================
print("="*60)
print("ПУНКТ 3.2: Очистка текста из wikitext")
print("="*60)

def clean_wikitext_symbols(chunk):
    """
    Заменяем специфичные для wikitext символы на обычные,
    а также убираем лишние пробелы.
    """
    chunk = chunk.replace('@-@', '-')
    chunk = chunk.replace('@.@', '.')
    chunk = chunk.replace('@,@', ',')
    chunk = ' '.join(chunk.split())  # нормализация пробелов
    return chunk

print(f"Объектов до очистки: {len(chunks)}")

# Применяем очистку и фильтрацию по качеству текста (is_good_text)
cleaned_wikitext = [clean_wikitext_symbols(c) for c in chunks]
cleaned_wikitext = [c for c in cleaned_wikitext if is_good_text(c)]

print(f"Объектов после очистки: {len(cleaned_wikitext)}")
print(f"Количество удалённых объектов: {len(chunks) - len(cleaned_wikitext)}")

# Сохраняем результат очистки в файл
output_path = "/content/drive/MyDrive/CommonCrawl/wikitext_cleaned.txt"
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(cleaned_wikitext))

print(f"Промежуточный результат сохранён по адресу: {output_path}")

ПУНКТ 3.2: Очистка текста из wikitext
Объектов до очистки: 5487
Объектов после очистки: 5337
Количество удалённых объектов: 150
Промежуточный результат сохранён по адресу: /content/drive/MyDrive/CommonCrawl/wikitext_cleaned.txt


In [ ]:
"""
Пункт 3.3 для wikitext: расчет энтропии и фильтрация дубликатов
"""

# ============================================================
# 1. Загрузка GPT-2 (независимо от предыдущих загрузок)
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

print("Загружаем модель и токенизатор GPT-2...")
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2_model.eval()
print("Модель GPT-2 успешно загружена.")

# ============================================================
# 2. Загрузка очищенного текстового датасета wikitext
# ============================================================
input_path = "/content/drive/MyDrive/CommonCrawl/wikitext_cleaned.txt"
with open(input_path, 'r', encoding='utf-8') as f:
    text = f.read()
cleaned_wikitext = [c.strip() for c in text.split('\n\n') if c.strip()]
print(f"Загружено объектов (абзацев): {len(cleaned_wikitext)}")

# ============================================================
# 3. Функция для пакетного расчёта энтропии
# ============================================================
def calculate_entropy_batched(texts, batch_size=4):
    entropies = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Подсчёт энтропии"):
        batch_texts = texts[i:i+batch_size]
        encodings = gpt2_tokenizer(batch_texts, return_tensors="pt", padding=True,
                                   truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            outputs = gpt2_model(**encodings)
            logits = outputs.logits
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = encodings["input_ids"][..., 1:].contiguous()
            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
            losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)),
                              shift_labels.view(-1))
            losses = losses.view(shift_labels.size(0), -1)
            mask = encodings["attention_mask"][..., 1:].contiguous()
            for j in range(len(batch_texts)):
                valid_losses = losses[j][mask[j] == 1]
                entropies.append(valid_losses.sum().item())
    return entropies

# ============================================================
# 4. Вычисление энтропии для всего поднабора
# ============================================================
sample_for_entropy = cleaned_wikitext
print(f"Вычисляем энтропию для {len(sample_for_entropy)} объектов...")

entropies = calculate_entropy_batched(sample_for_entropy, batch_size=4)

# ============================================================
# 5. Фильтрация по порогам энтропии
# ============================================================
entropies_np = np.array(entropies)
low_threshold = np.percentile(entropies_np, 5)
high_threshold = np.percentile(entropies_np, 95)

print(f"\nПорог низкой энтропии (5-й процентиль): {low_threshold:.2f}")
print(f"Порог высокой энтропии (95-й процентиль): {high_threshold:.2f}")

filtered_wikitext = [chunk for chunk, ent in zip(sample_for_entropy, entropies)
                     if low_threshold <= ent <= high_threshold]

# ============================================================
# 6. Расчет информационной плотности
# ============================================================
total_entropy = sum(entropies)
total_tokens = sum(len(gpt2_tokenizer.encode(chunk)) for chunk in sample_for_entropy)
rho_info = total_entropy / total_tokens

print(f"\nИнформационная плотность (rho_info): {rho_info:.4f} бит на токен")
print(f"Объектов после фильтрации осталось: {len(filtered_wikitext)}")

# ============================================================
# 7. Сохранение отфильтрованного датасета
# ============================================================
final_path = "/content/drive/MyDrive/CommonCrawl/wikitext_filtered.txt"
with open(final_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(filtered_wikitext))

file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\nФинальный датасет сохранён по адресу: {final_path}")
print(f"Размер файла: {file_size_mb:.2f} МБ")

Используем устройство: cuda
Загружаем модель и токенизатор GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Модель GPT-2 успешно загружена.
Загружено объектов (абзацев): 5337
Вычисляем энтропию для 5337 объектов...


Подсчёт энтропии: 100%|██████████| 1335/1335 [05:57<00:00,  3.73it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2874 > 1024). Running this sequence through the model will result in indexing errors



Порог низкой энтропии (5-й процентиль): 320.63
Порог высокой энтропии (95-й процентиль): 3267.08

Информационная плотность (rho_info): 3.3767 бит на токен
Объектов после фильтрации осталось: 4803

Финальный датасет сохранён по адресу: /content/drive/MyDrive/CommonCrawl/wikitext_filtered.txt
Размер файла: 8.52 МБ


In [ ]:
"""
Пункт 3.4 для wikitext: Сравнение трех токенизаторов в едином формате вывода
"""

# 1. Загружаем отфильтрованный wikitext
input_path = "/content/drive/MyDrive/CommonCrawl/wikitext_filtered.txt"
with open(input_path, 'r', encoding='utf-8') as f:
    wikitext_chunks = [c.strip() for c in f.read().split('\n\n') if c.strip()]

print(f"Загружено объектов из wikitext: {len(wikitext_chunks)}\n")

# 2. Фиксируем seed и выбираем случайный объект для всех токенизаторов
random.seed(42)
random_chunk = random.choice(wikitext_chunks)

divider = "=" * 80
print(divider)
print("СЛУЧАЙНЫЙ ОБЪЕКТ (первые 200 символов):")
print(divider)
print(f"'{random_chunk[:200]}...'")
print(f"Длина объекта: {len(random_chunk)} символов, {len(random_chunk.split())} слов\n")

# ============================================================
# CHAR-LEVEL ТОКЕНИЗАЦИЯ
# ============================================================
wiki_chars = set(''.join(wikitext_chunks))
char_vocab = ['<PAD>', '<UNK>'] + sorted(wiki_chars)
char_to_idx = {ch: idx for idx, ch in enumerate(char_vocab)}
idx_to_char = {idx: ch for idx, ch in enumerate(char_vocab)}

char_tokens = [char_to_idx.get(ch, char_to_idx['<UNK>']) for ch in random_chunk]
char_decoded = ''.join(idx_to_char[i] for i in char_tokens)
char_oov = sum(1 for ch in random_chunk if ch not in char_to_idx)

print(divider)
print("1) CHAR-LEVEL ТОКЕНИЗАТОР (обучен на wikitext)")
print(divider)
print(f"Размер словаря:           {len(char_vocab)}")
print(f"Длина последовательности: {len(char_tokens)}")
print(f"OOV (неизвестных символов): {char_oov} (по определению 0)")
print(f"Декодирование совпадает с оригиналом: {char_decoded == random_chunk}")
print(f"Первые 30 токенов (ID):    {char_tokens[:30]}")
print(f"Первые 30 токенов (текст): {[idx_to_char[i] for i in char_tokens[:30]]}")

# ============================================================
# WORD-LEVEL ТОКЕНИЗАЦИЯ
# ============================================================
all_words = [w for chunk in wikitext_chunks for w in chunk.lower().split()]
word_counts = Counter(all_words)
word_vocab_list = [w for w, c in word_counts.items() if c >= 2]
word_vocab = ['<PAD>', '<UNK>'] + sorted(word_vocab_list)
word_to_idx = {w: i for i, w in enumerate(word_vocab)}
idx_to_word = {i: w for i, w in enumerate(word_vocab)}

words = random_chunk.lower().split()
word_tokens = [word_to_idx.get(w, word_to_idx['<UNK>']) for w in words]
word_oov = sum(1 for w in words if w not in word_to_idx)
word_oov_ratio = 100 * word_oov / len(words)

print(f"\n{divider}")
print("2) WORD-LEVEL ТОКЕНИЗАТОР (обучен на wikitext)")
print(divider)
print(f"Размер словаря:           {len(word_vocab)}")
print(f"Длина последовательности: {len(word_tokens)}")
print(f"OOV (неизвестных слов):    {word_oov} из {len(words)} ({word_oov_ratio:.2f}%)")
print(f"Первые 20 токенов (ID):    {word_tokens[:20]}")
print(f"Первые 20 токенов (текст): {[idx_to_word[i] for i in word_tokens[:20]]}")

# ============================================================
# BPE ТОКЕНИЗАЦИЯ (на Common Crawl)
# ============================================================
print(f"\n{divider}")
print("3) BPE-ТОКЕНИЗАТОР (обучен на Common Crawl, без дообучения)")
print(divider)

try:
    _ = tokenizer.get_vocab_size()
    print("BPE-токенизатор найден в памяти.")
except NameError:
    print("BPE-токенизатор не найден. Обучаем заново на Common Crawl...")
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    with open("/content/drive/MyDrive/CommonCrawl/filtered_dataset_v2.txt", 'r', encoding='utf-8') as f:
        cc_text = f.read()
    cc_chunks = [c.strip() for c in cc_text.split('\n\n') if c.strip()]

    tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=10000, special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"], min_frequency=2)
    tokenizer.train_from_iterator(cc_chunks, trainer=trainer)

encoding = tokenizer.encode(random_chunk)
bpe_oov = encoding.tokens.count('<UNK>')

print(f"Размер словаря:           {tokenizer.get_vocab_size()}")
print(f"Длина последовательности: {len(encoding.ids)}")
print(f"OOV (неизвестных токенов): {bpe_oov}")
print(f"Первые 30 токенов (ID):    {encoding.ids[:30]}")
print(f"Первые 30 токенов (текст): {encoding.tokens[:30]}")

# ============================================================
# ОБЩАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# ============================================================
print(f"\n{divider}")
print("СВОДНАЯ ТАБЛИЦА ПО ТОКЕНИЗАТОРАМ")
print(divider)

header_fmt = "{:<15}{:<20}{:<22}{:<12}"
data_fmt = "{:<15}{:<20}{:<22}{:<12}"

print(header_fmt.format("Метод", "Размер словаря", "Длина последовательности", "OOV"))
print("-" * 70)
print(data_fmt.format("Char-level", len(char_vocab), len(char_tokens), "0 (по определ.)"))
print(data_fmt.format("Word-level", len(word_vocab), len(word_tokens), f"{word_oov_ratio:.2f}%"))
print(data_fmt.format("BPE (CC)", tokenizer.get_vocab_size(), len(encoding.ids), bpe_oov))

Загружено объектов из wikitext: 4803

СЛУЧАЙНЫЙ ОБЪЕКТ (первые 200 символов):
'New York State Route 368 ( NY 368 ) was a state highway in Onondaga County , New York , in the United States . It was one of the shortest routes in the county , extending for only 1 . 69 miles ( 2 . 7...'
Длина объекта: 480 символов, 105 слов

1) CHAR-LEVEL ТОКЕНИЗАТОР (обучен на wikitext)
Размер словаря:           997
Длина последовательности: 480
OOV (неизвестных символов): 0 (по определению 0)
Декодирование совпадает с оригиналом: True
Первые 30 токенов (ID):    [48, 71, 89, 2, 59, 81, 84, 77, 2, 53, 86, 67, 86, 71, 2, 52, 81, 87, 86, 71, 2, 21, 24, 26, 2, 10, 2, 48, 59, 2]
Первые 30 токенов (текст): ['N', 'e', 'w', ' ', 'Y', 'o', 'r', 'k', ' ', 'S', 't', 'a', 't', 'e', ' ', 'R', 'o', 'u', 't', 'e', ' ', '3', '6', '8', ' ', '(', ' ', 'N', 'Y', ' ']

2) WORD-LEVEL ТОКЕНИЗАТОР (обучен на wikitext)
Размер словаря:           35773
Длина последовательности: 105
OOV (неизвестных слов):    0 из 105 (0.00%)
Первы

In [ ]:
"""
Пункт 3.5: Packed Batching для токенов wikitext
"""
from collections import defaultdict

# 1. Токенизируем весь wikitext с помощью BPE (Common Crawl)
print("Токенизация всего wikitext с использованием BPE...")

all_token_ids = [tokenizer.encode(chunk).ids for chunk in wikitext_chunks]

total_tokens = sum(len(ids) for ids in all_token_ids)
num_objects = len(all_token_ids)

print(f"Общее количество объектов: {num_objects}")
print(f"Общее число токенов:      {total_tokens}")
print(f"Средняя длина объекта:    {total_tokens / num_objects:.1f} токенов")
print(f"Минимальная длина:        {min(len(ids) for ids in all_token_ids)} токенов")
print(f"Максимальная длина:       {max(len(ids) for ids in all_token_ids)} токенов")

# ============================================================
# 2. Функция packed batching
# ============================================================
def packed_batching(token_sequences, batch_size, pad_token_id=0):
    """
    Упаковываем последовательности в батчи фиксированной длины.

    Возвращает:
        packed_batches: список списков токенов длины batch_size
        attention_masks: список масок с номерами объектов (0 для PAD)
    """
    packed_batches = []
    attention_masks = []

    current_batch = []
    current_mask = []
    current_obj_id = 1  # Счётчик объектов в батче

    for tokens in token_sequences:
        idx = 0
        while idx < len(tokens):
            space_left = batch_size - len(current_batch)
            segment = tokens[idx: idx + space_left]

            current_batch.extend(segment)
            current_mask.extend([current_obj_id] * len(segment))

            idx += space_left
            current_obj_id += 1

            if len(current_batch) == batch_size:
                packed_batches.append(current_batch)
                attention_masks.append(current_mask)
                current_batch = []
                current_mask = []
                current_obj_id = 1

    # Дополняем последний батч PADами, если нужно
    if current_batch:
        pad_len = batch_size - len(current_batch)
        current_batch.extend([pad_token_id] * pad_len)
        current_mask.extend([0] * pad_len)
        packed_batches.append(current_batch)
        attention_masks.append(current_mask)

    return packed_batches, attention_masks


# ============================================================
# 3. Выполняем packed batching
# ============================================================
BATCH_SIZE = 512  # длина последовательности

print(f"\nВыполняем packed batching (batch_size={BATCH_SIZE})...")
packed_batches, attention_masks = packed_batching(all_token_ids, batch_size=BATCH_SIZE)

total_batch_len = sum(len(b) for b in packed_batches)
pad_tokens = total_batch_len - total_tokens
efficiency = total_tokens / total_batch_len * 100

print("\n--- Итоги Packed Batching ---")
print(f"Количество батчей:     {len(packed_batches)}")
print(f"Общая длина батчей:    {total_batch_len} токенов")
print(f"Полезных токенов:      {total_tokens}")
print(f"PAD-токенов:           {pad_tokens}")
print(f"Эффективность упаковки: {efficiency:.2f}%")


# ============================================================
# 4. Демонстрация первого и последнего батча
# ============================================================
print("\n--- Пример первого батча (первые 50 токенов) ---")
print(f"Токены:              {packed_batches[0][:50]}")
print(f"Маска внимания:      {attention_masks[0][:50]}")

print("\nГраницы объектов в первом батче (позиции начала):")
prev_mask = 0
for pos, mask_val in enumerate(attention_masks[0]):
    if mask_val != prev_mask and mask_val != 0:
        print(f"  Позиция {pos}: начало объекта #{mask_val}")
    prev_mask = mask_val

print("\n--- Пример последнего батча (последние 50 токенов) ---")
print(f"Токены:              {packed_batches[-1][-50:]}")
print(f"Маска внимания:      {attention_masks[-1][-50:]}")

Токенизация всего wikitext с использованием BPE...
Общее количество объектов: 4803
Общее число токенов:      2288250
Средняя длина объекта:    476.4 токенов
Минимальная длина:        73 токенов
Максимальная длина:       3094 токенов

Выполняем packed batching (batch_size=512)...

--- Итоги Packed Batching ---
Количество батчей:     4470
Общая длина батчей:    2288640 токенов
Полезных токенов:      2288250
PAD-токенов:           390
Эффективность упаковки: 99.98%

--- Пример первого батча (первые 50 токенов) ---
Токены:              [3122, 60, 1, 299, 2508, 1093, 2605, 15, 22, 541, 10159, 771, 15811, 161, 449, 7, 6945, 22, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 15, 9, 10588, 11, 2508, 1093, 2605, 188, 173, 11914, 2849, 15, 8, 9, 7659, 338, 6438, 1116, 168, 160, 2508, 1093]
Маска внимания:      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Границы объектов в первом батче (позиции начала):
  П

In [ ]:
# ============================================================
# Сохраняем упакованные батчи и соответствующие маски в сжатом формате .npz
# ============================================================
save_path = "/content/drive/MyDrive/CommonCrawl/wikitext_packed_batches.npz"

np.savez_compressed(
    save_path,
    batches=np.array(packed_batches, dtype=np.int32),
    masks=np.array(attention_masks, dtype=np.int8),
    batch_size=BATCH_SIZE
)

print(f"Файл успешно сохранён по пути: {save_path}")
file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"Размер файла: {file_size_mb:.2f} MB")

Файл успешно сохранён по пути: /content/drive/MyDrive/CommonCrawl/wikitext_packed_batches.npz
Размер файла: 3.16 MB
